### https://www.kaggle.com/competitions/drawing-with-llms

In [25]:
import kagglehub
import pandas as pd
import re

In [26]:
import openai
import pandas as pd
import time
import os
from dotenv import load_dotenv
load_dotenv()
openai_api_key = os.getenv("OPENAI_API_KEY")
deepseek_api_key = os.getenv("DEEPSEEK_API_KEY")
gemini_api_key=os.getenv("GEMINI_API_KEY")

In [18]:
# from openai import OpenAI
# client = OpenAI(api_key=deepseek_api_key, base_url="https://api.deepseek.com")

# response = client.chat.completions.create(
#     model="deepseek-chat",
#     messages=[
#         {"role": "system", "content": "You are a helpful assistant"},
#         {"role": "user", "content": "Hello are you"},
#     ],
#     stream=False
# )

# print(response.choices[0].message.content)

In [19]:
# # Function to get summary from OpenAI API
# def get_svg_code(text):
#     instruction = f"""
#             Generate SVG code to visually represent the following text description, while respecting the given constraints.
#             <constraints>
#             * **Allowed Elements:** `svg`, `path`, `circle`, `rect`, `ellipse`, `line`, `polyline`, `polygon`, `g`, `linearGradient`, `radialGradient`, `stop`, `defs`
#             * **Allowed Attributes:** `viewBox`, `width`, `height`, `fill`, `stroke`, `stroke-width`, `d`, `cx`, `cy`, `r`, `x`, `y`, `rx`, `ry`, `x1`, `y1`, `x2`, `y2`, `points`, `transform`, `opacity`
#             </constraints>

#             Please ensure that the generated SVG code is well-formed, valid, and strictly adheres to these constraints. 
#             Focus on a clear and concise representation of the input description within the given limitations. 
#             Always give the complete SVG code with nothing omitted. Never use an ellipsis.

#             The code is scored based on similarity to the description, Visual question anwering and aesthetic components.
#             Please generate a svg code considering similarity, visual question answering and aesthetic compoenents.

#             input description: {text}
#             """

#     try:
#         response = client.chat.completions.create(
#             model="deepseek-chat",
#             messages=[
#                 {"role": "system", "content": "Generate a SVG code as per instruction"},
#                 {"role": "user", "content": instruction}
#             ],
#             temperature=0.7,
#             max_tokens=3200
#         )
#         return response.choices[0].message.content.strip()
    
#     except Exception as e:
#         print(f"Error: {e}")
#         return None

In [20]:
# from tqdm import tqdm
# tqdm.pandas()
# df["deepseek_response"] = df.progress_apply(lambda row: get_svg_code(row["description"]), axis=1)

In [22]:
from google import genai
client = genai.Client(api_key=gemini_api_key)

# Function to get summary from OpenAI API
def get_svg_code_gemini(text):
    instruction = f"""
            Generate SVG code to visually represent the following text description, while respecting the given constraints.
            <constraints>
            * **Allowed Elements:** `svg`, `path`, `circle`, `rect`, `ellipse`, `line`, `polyline`, `polygon`, `g`, `linearGradient`, `radialGradient`, `stop`, `defs`
            * **Allowed Attributes:** `viewBox`, `width`, `height`, `fill`, `stroke`, `stroke-width`, `d`, `cx`, `cy`, `r`, `x`, `y`, `rx`, `ry`, `x1`, `y1`, `x2`, `y2`, `points`, `transform`, `opacity`
            </constraints>

            Please ensure that the generated SVG code is well-formed, valid, and strictly adheres to these constraints. 
            Focus on a clear and concise representation of the input description within the given limitations. 
            Always give the complete SVG code with nothing omitted. Never use an ellipsis.

            The code is scored based on similarity to the description, Visual question anwering and aesthetic components.
            Please generate a detailed svg code accordingly.
            
            input description: {text}
            """

    try:
        response = client.models.generate_content(
            model="gemini-2.0-flash", contents= instruction
        )
        return response
    
    except Exception as e:
        print(f"Error: {e}")
        return None



In [1]:
import pandas as pd
from tqdm import tqdm
import os

df = pd.read_csv('./drawing-with-llms/gemini_20_description_master_48k.csv')
df=df.iloc[0:10000]
tqdm.pandas()

batch_size = 100
total_rows = len(df)

# Optional: directory to store batches
os.makedirs("batches", exist_ok=True)

for i in range(0, total_rows, batch_size):
    batch_num = i // batch_size + 1
    filename = f"batches/response_batch_{batch_num}.csv"
    
    if os.path.exists(filename):
        print(f"Skipping batch {batch_num}, already exists.")
        continue

    batch_df = df.iloc[i:i+batch_size].copy()
    
    # Apply your function here
    batch_df["gemini_response"] = batch_df.progress_apply(
        lambda row: get_svg_code_gemini(row["description"]), axis=1
    )

    # Save after processing
    batch_df.to_csv(filename, index=False)
    print(f"Saved batch {batch_num} to {filename}")


In [13]:
# from google import genai
# client = genai.Client(api_key=gemini_api_key)

# # # Function to get summary from OpenAI API
# def get_topic():
#     instruction = f"""
#     I am participating in an SVG code generation competition.
    
#     The competition involves generating SVG images based on short textual descriptions of everyday objects and scenes, spanning a wide range of categories. The key guidelines are as follows:
    
#     - Descriptions are generic and do not contain brand names, trademarks, or personal names.
#     - No descriptions include people, even in generic terms.
#     - Descriptions are concise—each is no more than 200 characters, with an average length of about 50 characters.
#     - Categories cover various domains, with some overlap between public and private test sets.
    
#     To train a small LLM model, I am preparing a synthetic dataset. Could you generate **500 unique topics** aligned with the competition style?
    
#     **Requirements:**
#     - Each topic should range between **20 and 200 characters**, with an **average around 60 characters**.
#     - Ensure **diversity and creativity** across topics.
#     - **50% of the topics** should come from the categories of **landscapes**, **abstract art**, and **fashion**.
#     - Avoid duplication or overly similar phrasing.
    
#     **Example topics:**
#     <example>
#     a purple forest at dusk, gray wool coat with a faux fur collar,  
#     a lighthouse overlooking the ocean, burgundy corduroy pants with patch pockets and silver buttons,  
#     orange corduroy overalls, a purple silk scarf with tassel trim,  
#     a green lagoon under a cloudy sky, crimson rectangles forming a chaotic grid,  
#     purple pyramids spiraling around a bronze cone, magenta trapezoids layered on a translucent silver sheet,  
#     a snowy plain, black and white checkered pants,  
#     a starlit night over snow-covered peaks, khaki triangles and azure crescents,  
#     a maroon dodecahedron interwoven with teal threads.
#     </example>
    
#     Please return the 100 topics in csv format.
#     """


#     try:
#         response = client.models.generate_content(
#             model="gemini-2.0-flash", contents= instruction
#         )
#         return response
    
#     except Exception as e:
#         print(f"Error: {e}")
#         return None

In [32]:
# for i in range(101, 200):
#     print('running:',i)
#     response = get_topic_gemini()
#     filename = f"response_{i}.txt"
#     with open(filename, "w", encoding="utf-8") as f:
#         f.write(response.text)